In [ ]:
from pathlib import Path
import zipfile
from google.colab import drive

drive.mount("/content/drive")

# Update these two paths to match your Google Drive layout
ZIP_PATH = Path("/content/drive/MyDrive/paddy_disease_classification.zip")
CHECKPOINT_DIR = Path("/content/drive/MyDrive/paddy_efficientnet_b4_ckpts")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

EXTRACT_DIR = Path("/content/paddy_data")
if not EXTRACT_DIR.exists():
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(EXTRACT_DIR)
    print("Extraction complete.")
else:
    print("Dataset already extracted.")

# Auto-locate train.csv in case the zip contains a nested root folder
csv_candidates = list(EXTRACT_DIR.rglob("train.csv"))
assert len(csv_candidates) > 0, f"train.csv not found under {EXTRACT_DIR}"

CSV_PATH = csv_candidates[0]
DATA_ROOT = CSV_PATH.parent
IMG_DIR = DATA_ROOT / "train_images"

assert IMG_DIR.is_dir(), f"Expected train_images folder at {IMG_DIR}"
print(f"Found CSV: {CSV_PATH}")
print(f"Found images root: {IMG_DIR}")

In [ ]:
import random
import numpy as np
import pandas as pd
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from sklearn.model_selection import train_test_split

# --- Config ---
SEED = 42
IMG_SIZE = 380           # Native resolution for EfficientNet-B4
BATCH_SIZE = 32          # Paper Table 3 setting
NUM_WORKERS = 2          # Standard Colab CPU core count

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# --- Load & Encode Labels ---
df = pd.read_csv(CSV_PATH)
class_names = sorted(df["label"].unique().tolist())
label_to_idx = {name: idx for idx, name in enumerate(class_names)}
idx_to_label = {idx: name for name, idx in label_to_idx.items()}
df["target"] = df["label"].map(label_to_idx)

# Build full file paths: train_images/<label>/<image_id>
df["filepath"] = df.apply(
    lambda row: str(IMG_DIR / row["label"] / row["image_id"]), axis=1
)

# Verify paths exist (spot-check first row)
assert Path(df["filepath"].iloc[0]).is_file(), f"Missing image: {df['filepath'].iloc[0]}"

# Stratified 75% Train / 12.5% Val / 12.5% Test split
train_df, temp_df = train_test_split(
    df, test_size=0.25, stratify=df["target"], random_state=SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["target"], random_state=SEED
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"Split sizes -> Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")
print("Class index mapping:", label_to_idx)

# --- Transforms (ImageNet normalization) ---
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(degrees=20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

eval_transforms = transforms.Compose([
    transforms.Resize(int(IMG_SIZE * 1.05)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# --- Dataset ---
class PaddyDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, transform=None):
        self.filepaths = dataframe["filepath"].values
        self.targets = dataframe["target"].values
        self.transform = transform

    def __len__(self):
        return len(self.filepaths)

    def __getitem__(self, idx):
        img = Image.open(self.filepaths[idx]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(self.targets[idx], dtype=torch.long)

# --- Leak-Free Balanced Sampling on Train Split ---
class_counts = train_df["target"].value_counts().sort_index().values
class_weights = 1.0 / class_counts
sample_weights = torch.tensor(
    [class_weights[t] for t in train_df["target"].values], dtype=torch.double
)

train_sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(train_df),  # Set higher (e.g., 15000) if you want longer epochs
    replacement=True,
)

train_dataset = PaddyDataset(train_df, transform=train_transforms)
val_dataset = PaddyDataset(val_df, transform=eval_transforms)
test_dataset = PaddyDataset(test_df, transform=eval_transforms)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=train_sampler,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=True,  # Prevents batch size = 1 issues with BatchNorm1d
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

In [ ]:
import torch.nn as nn
from torchvision.models import efficientnet_b4, EfficientNet_B4_Weights

def build_paddy_efficientnet_b4(
    num_classes: int = 10,
    unfreeze_from_stage: int = 4,  # 4 = freeze stages 0..3, train top 5 stages (4..8) + head
                                   # 0 = fine-tune entire backbone
) -> nn.Module:
    model = efficientnet_b4(weights=EfficientNet_B4_Weights.IMAGENET1K_V1)

    # Freeze all backbone weights first
    for param in model.features.parameters():
        param.requires_grad = False

    # Unfreeze from selected stage onward (EfficientNet-B4 has stages 0..8)
    for stage_idx in range(unfreeze_from_stage, len(model.features)):
        for param in model.features[stage_idx].parameters():
            param.requires_grad = True

    # Note: model.avgpool is already Global Average Pooling (AdaptiveAvgPool2d(1))
    in_features = model.classifier[1].in_features  # 1792 for EfficientNet-B4

    # Custom head matching Section III.D (GAP -> 1024 Dense + BN + Swish/SiLU -> 10 classes)
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.4),
        nn.Linear(in_features, 1024),
        nn.BatchNorm1d(1024),
        nn.SiLU(inplace=True),  # PyTorch's SiLU is Swish with beta=1
        nn.Dropout(p=0.3),
        nn.Linear(1024, num_classes),
    )
    return model

model = build_paddy_efficientnet_b4(num_classes=len(class_names), unfreeze_from_stage=4)
model = model.to(device)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable_params:,} / {total_params:,} ({100*trainable_params/total_params:.1f}%)")

In [ ]:
import time
from tqdm.auto import tqdm

EPOCHS = 50               # Paper Table 3 setting
LEARNING_RATE = 1e-4      # Paper Table 3 setting (Adam, 0.0001)
GRAD_ACCUM_STEPS = 1      # Set BATCH_SIZE=16 & GRAD_ACCUM_STEPS=2 if you hit CUDA OOM

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LEARNING_RATE,
)
scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))

best_ckpt_path = CHECKPOINT_DIR / "efficientnet_b4_paddy_best.pt"
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_val_acc = 0.0

def run_epoch(loader, is_train: bool):
    model.train() if is_train else model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    if is_train:
        optimizer.zero_grad(set_to_none=True)

    context = torch.enable_grad() if is_train else torch.no_grad()
    with context:
        for step, (images, targets) in enumerate(tqdm(loader, leave=False)):
            images = images.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)

            with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
                logits = model(images)
                loss = criterion(logits, targets)

            if is_train:
                scaled_loss = loss / GRAD_ACCUM_STEPS
                scaler.scale(scaled_loss).backward()

                if (step + 1) % GRAD_ACCUM_STEPS == 0 or (step + 1) == len(loader):
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad(set_to_none=True)

            batch_size = targets.size(0)
            running_loss += loss.item() * batch_size
            preds = logits.argmax(dim=1)
            correct += (preds == targets).sum().item()
            total += batch_size

    return running_loss / total, correct / total

start_time = time.time()
for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    train_loss, train_acc = run_epoch(train_loader, is_train=True)
    val_loss, val_acc = run_epoch(val_loader, is_train=False)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    saved_flag = ""
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_acc": best_val_acc,
                "class_names": class_names,
            },
            best_ckpt_path,
        )
        saved_flag = " * [Saved Best]"

    elapsed = time.time() - t0
    print(
        f"Epoch {epoch:02d}/{EPOCHS} ({elapsed:.1f}s) | "
        f"Train Loss: {train_loss:.4f}, Acc: {train_acc*100:.2f}% | "
        f"Val Loss: {val_loss:.4f}, Acc: {val_acc*100:.2f}%{saved_flag}"
    )

print(f"\nTraining finished in {(time.time() - start_time)/60:.1f} min. Best Val Acc: {best_val_acc*100:.2f}%")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_curve,
    auc,
)
from sklearn.preprocessing import label_binarize

# --- 1. Plot Training Curves (Paper Fig. 5) ---
epochs_range = range(1, len(history["train_loss"]) + 1)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(epochs_range, history["train_loss"], label="Train")
axes[0].plot(epochs_range, history["val_loss"], label="Validation")
axes[0].set_title("Model Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(epochs_range, history["train_acc"], label="Train")
axes[1].plot(epochs_range, history["val_acc"], label="Validation")
axes[1].set_title("Model Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# --- 2. Load Best Checkpoint & Run Test Inference ---
ckpt = torch.load(best_ckpt_path, map_location=device, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()
print(f"Loaded best checkpoint from epoch {ckpt['epoch']} (Val Acc: {ckpt['val_acc']*100:.2f}%)\n")

all_targets = []
all_preds = []
all_probs = []

with torch.no_grad():
    for images, targets in tqdm(test_loader, desc="Testing"):
        images = images.to(device, non_blocking=True)
        with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
            logits = model(images)
            probs = torch.softmax(logits, dim=1)

        all_probs.append(probs.cpu().numpy())
        all_preds.append(probs.argmax(dim=1).cpu().numpy())
        all_targets.append(targets.numpy())

y_true = np.concatenate(all_targets)
y_pred = np.concatenate(all_preds)
y_prob = np.concatenate(all_probs)

# --- 3. Classification Report (Paper Fig. 6) ---
print("Test Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

# --- 4. Confusion Matrix & Multi-Class ROC Curves (Paper Fig. 4) ---
cm = confusion_matrix(y_true, y_pred)
y_true_bin = label_binarize(y_true, classes=list(range(len(class_names))))

fig, axes = plt.subplots(1, 2, figsize=(16, 6.5))

# Confusion Matrix
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
    ax=axes[0],
)
axes[0].set_title("Test Confusion Matrix")
axes[0].set_xlabel("Predicted Label")
axes[0].set_ylabel("True Label")
axes[0].tick_params(axis="x", rotation=45)

# Per-class + Micro/Macro ROC
fpr, tpr, roc_auc = {}, {}, {}
for i, name in enumerate(class_names):
    fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], y_prob[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])
    axes[1].plot(fpr[i], tpr[i], lw=1.5, label=f"{name} (AUC = {roc_auc[i]:.3f})")

# Micro-average ROC
fpr["micro"], tpr["micro"], _ = roc_curve(y_true_bin.ravel(), y_prob.ravel())
roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

# Macro-average ROC
all_fpr = np.unique(np.concatenate([fpr[i] for i in range(len(class_names))]))
mean_tpr = np.zeros_like(all_fpr)
for i in range(len(class_names)):
    mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
mean_tpr /= len(class_names)
roc_auc["macro"] = auc(all_fpr, mean_tpr)

axes[1].plot(
    fpr["micro"], tpr["micro"],
    linestyle=":", lw=2.5, label=f"micro-average (AUC = {roc_auc['micro']:.3f})"
)
axes[1].plot(
    all_fpr, mean_tpr,
    linestyle="--", lw=2.5, label=f"macro-average (AUC = {roc_auc['macro']:.3f})"
)
axes[1].plot([0, 1], [0, 1], "k--", lw=1)
axes[1].set_title("Test ROC Curves (One-vs-Rest)")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].legend(loc="lower right", fontsize=8)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
!pip install -q onnx onnxscript onnxruntime

import json
from pathlib import Path
import numpy as np
import onnxruntime as ort
import torch
import torch.nn as nn
from torchvision.models import efficientnet_b4
from google.colab import drive

drive.mount("/content/drive")

# Make sure this matches the folder path you used during training
CHECKPOINT_DIR = Path("/content/drive/MyDrive/paddy_efficientnet_b4_ckpts")
best_ckpt_path = CHECKPOINT_DIR / "efficientnet_b4_paddy_best.pt"
assert best_ckpt_path.is_file(), f"Checkpoint not found at {best_ckpt_path}"

def build_paddy_efficientnet_b4(num_classes: int = 10) -> nn.Module:
    # weights=None skips downloading ImageNet weights since your checkpoint overwrites them
    model = efficientnet_b4(weights=None)
    in_features = model.classifier[1].in_features  # 1792
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.4),
        nn.Linear(in_features, 1024),
        nn.BatchNorm1d(1024),
        nn.SiLU(inplace=True),
        nn.Dropout(p=0.3),
        nn.Linear(1024, num_classes),
    )
    return model

ckpt = torch.load(best_ckpt_path, map_location="cpu", weights_only=False)
model_cpu = build_paddy_efficientnet_b4(num_classes=len(ckpt["class_names"]))
model_cpu.load_state_dict(ckpt["model_state_dict"])
model_cpu.eval()

dummy_input = torch.randn(1, 3, 380, 380)
onnx_path = CHECKPOINT_DIR / "efficientnet_b4_paddy.onnx"
labels_path = CHECKPOINT_DIR / "class_names.json"

torch.onnx.export(
    model_cpu,
    (dummy_input,),
    str(onnx_path),
    dynamo=True,
    opset_version=18,
    external_data=False,
    input_names=["input"],
    output_names=["logits"],
)

with open(labels_path, "w") as f:
    json.dump(ckpt["class_names"], f, indent=2)

# Verify PyTorch vs ONNX Runtime outputs match
with torch.no_grad():
    pt_out = model_cpu(dummy_input).numpy()

sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
ort_out = sess.run(None, {"input": dummy_input.numpy()})[0]
max_diff = np.max(np.abs(pt_out - ort_out))

print(f"Loaded checkpoint from Epoch {ckpt['epoch']} (Val Acc: {ckpt['val_acc']*100:.2f}%)")
print(f"Exported ONNX model ({onnx_path.stat().st_size / 1e6:.1f} MB): {onnx_path}")
print(f"Saved labels to: {labels_path}")
print(f"Max PyTorch vs ONNX diff: {max_diff:.2e}")

In [ ]:
!pip install -q executorch

import json
from pathlib import Path
import torch
import torch.nn as nn
from torch.export import export
from torchvision.models import efficientnet_b4
from executorch.backends.xnnpack.partition.xnnpack_partitioner import XnnpackPartitioner
from executorch.exir import to_edge_transform_and_lower
from executorch.runtime import Runtime
from google.colab import drive

drive.mount("/content/drive")

CHECKPOINT_DIR = Path("/content/drive/MyDrive/paddy_efficientnet_b4_ckpts")
best_ckpt_path = CHECKPOINT_DIR / "efficientnet_b4_paddy_best.pt"
assert best_ckpt_path.is_file(), f"Checkpoint not found at {best_ckpt_path}"

def build_paddy_efficientnet_b4(num_classes: int = 10) -> nn.Module:
    model = efficientnet_b4(weights=None)
    in_features = model.classifier[1].in_features  # 1792
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.4),
        nn.Linear(in_features, 1024),
        nn.BatchNorm1d(1024),
        nn.SiLU(inplace=True),
        nn.Dropout(p=0.3),
        nn.Linear(1024, num_classes),
    )
    return model

ckpt = torch.load(best_ckpt_path, map_location="cpu", weights_only=False)
model_cpu = build_paddy_efficientnet_b4(num_classes=len(ckpt["class_names"]))
model_cpu.load_state_dict(ckpt["model_state_dict"])
model_cpu.eval()

example_inputs = (torch.randn(1, 3, 380, 380),)

# Export -> Lower to XNNPACK -> Serialize to .pte
exported_program = export(model_cpu, example_inputs)
et_program = to_edge_transform_and_lower(
    exported_program,
    partitioner=[XnnpackPartitioner()],
).to_executorch()

pte_path = CHECKPOINT_DIR / "efficientnet_b4_paddy.pte"
labels_path = CHECKPOINT_DIR / "class_names.json"

with open(pte_path, "wb") as f:
    f.write(et_program.buffer)

with open(labels_path, "w") as f:
    json.dump(ckpt["class_names"], f, indent=2)

# Verify PyTorch vs ExecuTorch output parity
with torch.no_grad():
    pt_out = model_cpu(*example_inputs)

et_runtime = Runtime.get()
program = et_runtime.load_program(pte_path)
forward_method = program.load_method("forward")
et_out = forward_method.execute(example_inputs)[0]

max_diff = (pt_out - et_out).abs().max().item()
print(f"Exported ExecuTorch model ({pte_path.stat().st_size / 1e6:.1f} MB): {pte_path}")
print(f"Saved labels: {labels_path}")
print(f"Max PyTorch vs ExecuTorch diff: {max_diff:.2e}")

In [ ]:
!pip install -q "git+https://github.com/huggingface/optimum-executorch.git" torchao transformers accelerate

from pathlib import Path
from transformers import AutoTokenizer
from google.colab import drive

drive.mount("/content/drive")

LOCAL_MODEL_DIR = Path("/content/drive/MyDrive/llama_model_1b")
LLAMA_OUT_DIR = Path("/content/drive/MyDrive/paddy_efficientnet_b4_ckpts/llama32_1b_et")
LLAMA_OUT_DIR.mkdir(parents=True, exist_ok=True)

# Sanity-check required 1B files before exporting
required = [
    "config.json",
    "model.safetensors",
    "tokenizer.json",
    "tokenizer_config.json",
]
missing = [f for f in required if not (LOCAL_MODEL_DIR / f).is_file()]
assert not missing, f"Missing files in {LOCAL_MODEL_DIR}: {missing}"
print("All required Llama 1B files found. Starting ExecuTorch export...")

!optimum-cli export executorch \
    --model "{LOCAL_MODEL_DIR}" \
    --task "text-generation" \
    --recipe "xnnpack" \
    --use_custom_sdpa \
    --use_custom_kv_cache \
    --qlinear 8da4w \
    --qembedding 8w \
    --max_seq_len 2048 \
    --output_dir "{LLAMA_OUT_DIR}"

# Copy tokenizer files into the exported bundle so the Pi runs 100% offline
tokenizer = AutoTokenizer.from_pretrained(str(LOCAL_MODEL_DIR), local_files_only=True)
tokenizer.save_pretrained(LLAMA_OUT_DIR)
print(f"Saved Llama 3.2 1B ExecuTorch bundle to: {LLAMA_OUT_DIR}")